# preRA cohort scRNA analysis in Python for Monocytes 
- CertPro

In [ ]:
import h5py
import scipy.sparse as scs
import pandas as pd
import anndata
import os
import glob
from matplotlib import pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import median_abs_deviation
import scanpy as sc
# import sc_toolbox as sct
import decoupler as dc
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42

In [ ]:
# define some color patterns for plotting
nejm_color = ["#BC3C29FF", "#0072B5FF", "#E18727FF", "#20854EFF", "#7876B1FF", "#6F99ADFF", "#FFDC91FF", "#EE4C97FF"]
jama_color = ["#374E55FF", "#DF8F44FF", "#00A1D5FF", "#B24745FF", "#79AF97FF", "#6A6599FF", "#80796BFF"]

In [ ]:
# define working path
data_path = '/home/jupyter/data/ra_longitudinal/scrna/certPro/'
fig_path = '/home/jupyter/data/ra_longitudinal/figures/mono/'
meta_path = '/home/jupyter/github/ra-longitudinal/metadata/'
output_path = '/home/jupyter/data/ra_longitudinal/output_results/cd16mono/'
# define a project name
proj_name = 'RA_lg_converters_scRNA_mono_'
# sc.set_figure_params(fig_path)
sc.settings.figdir = fig_path
sc.settings.autosave=False
sc.set_figure_params(vector_friendly=True, dpi_save=300)

In [ ]:
# set fig size
plt.rcParams['figure.figsize'] = [10, 8]

# load data

In [ ]:
# load the deep clean data
joint_adata_fl = sc.read_h5ad(
    '/home/jupyter/data/ra_longitudinal/scrna/certPro/ALTRA_certPro_scRNA_141_samples_combined_adata.h5ad'
)

In [ ]:
joint_adata_fl.obs['sample.sampleKitGuid'].unique

In [ ]:
# subset the monocytes
joint_adata_fl.obs.loc[joint_adata_fl.obs['AIFI_L3_new'].str.contains('monocyte'), 'AIFI_L3_new'].unique()

In [ ]:
# subset aim3 data
aim3_meta = pd.read_csv(meta_path + 'ALTRA_RA_Aim3_ALTRA_converters_longitudinal_scrna_metadata.csv')
aim3_meta.columns

In [ ]:
# subset the monocytes in aim3
mono_adata = joint_adata_fl[(joint_adata_fl.obs['AIFI_L3_new'].str.contains('monocyte'))].copy()

In [ ]:
mono_adata

## Rerun basic normalization

In [ ]:
mono_adata.obs['sample.sampleKitGuid'].unique()

In [ ]:
# save the raw counts
mono_adata.layers['counts'] = mono_adata.X.copy()

In [ ]:
# mitochondrial genes
mono_adata.var["mt"] = mono_adata.var_names.str.startswith("MT-")
# ribosomal genes
mono_adata.var["ribo"] = mono_adata.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes
mono_adata.var["hb"] = mono_adata.var_names.str.contains(("^HB[^(P)]"))
sc.pp.calculate_qc_metrics(
    mono_adata, qc_vars=["mt", "ribo", "hb"], inplace=True, percent_top=[20], log1p=True
)

In [ ]:
# cpm normalization
sc.pp.normalize_total(mono_adata, target_sum=1e4, inplace=True)
sc.pp.log1p(mono_adata)

In [ ]:
# %%time
sc.pp.highly_variable_genes(mono_adata, min_mean=0.0125, max_mean=3, min_disp=0.5)

In [ ]:
mono_adata.raw = mono_adata

In [ ]:
mono_adata.raw.X

In [ ]:
sc.pp.scale(mono_adata, max_value=10)

In [ ]:
# setting highly variable as highly deviant to use scanpy 'use_highly_variable' argument in sc.pp.pca
sc.pp.pca(mono_adata, svd_solver="arpack", use_highly_variable=True)

In [ ]:
sc.pl.pca_scatter(mono_adata, color=["pct_counts_ribo", 'pct_counts_mt'])

In [ ]:
# # setting highly variable as highly deviant to use scanpy 'use_highly_variable' argument in sc.pp.pca
# sc.pp.pca(mono_adata, svd_solver="arpack", use_highly_variable=True)

In [ ]:
# plot the principle component variance explained
sc.pl.pca_variance_ratio(mono_adata, log=True)

In [ ]:
sc.pp.neighbors(mono_adata, n_neighbors=10, n_pcs=20, use_rep='X_pca')
sc.tl.umap(mono_adata)


In [ ]:
# save the pre-harmonaized umap
mono_adata.obsm['X_orignial_umap'] = mono_adata.obsm['X_umap'].copy()

In [ ]:
mono_adata

In [ ]:
# run harmony
import scanpy.external as sce
sce.pp.harmony_integrate(mono_adata, ['file.batchID', 'subject.biologicalSex'],
                         adjusted_basis='X_pca_harmony')
sc.pp.neighbors(mono_adata, n_neighbors=30, n_pcs=20, use_rep='X_pca_harmony')
sc.tl.umap(mono_adata)
mono_adata.obsm['X_harmony_umap'] = mono_adata.obsm['X_umap'].copy()

In [ ]:
# sc.pp.neighbors(mono_adata, n_neighbors=10, n_pcs=30, use_rep='X_pca_harmony',key_added='harmony_neighbors')
# sc.tl.umap(mono_adata, neighbors_key='harmony_neighbors')
# sc.tl.leiden(mono_adata, key_added="leiden_0_5", resolution=0.5, neighbors_key='harmony_neighbors')

### run TSNE

In [ ]:
# from openTSNE import TSNE
# tsne = TSNE(
#     perplexity=30,
#     metric="euclidean",
#     n_jobs=58,
#     random_state=42,
#     verbose=True,
# )

In [ ]:
# %%time 
# embedding_train = tsne.fit(mono_adata.obsm['X_pca_harmony'])

In [ ]:
# sc.pp.neighbors(mono_adata, n_neighbors=10, n_pcs=30, use_rep='X_pca_harmony')
sc.tl.tsne(mono_adata, use_rep='X_pca_harmony')

In [ ]:
mono_adata.obsm['X_pca_harmony']

In [ ]:
# mono_adata.obsm['X_harmony_umap'] = mono_adata.obsm['X_umap'].copy()

In [ ]:
sc.pl.pca_loadings(mono_adata, components = '1,2,3')

In [ ]:
# run clusters
sc.tl.leiden(mono_adata, key_added="leiden_0_5", resolution=0.5)

In [ ]:
# run clusters
sc.tl.leiden(mono_adata, key_added="leiden_1", resolution=1, n_iterations=2)

In [ ]:
# run clusters
sc.tl.leiden(mono_adata, key_added="leiden_1_2", resolution=1.2, n_iterations=2)

In [ ]:
mono_adata

In [ ]:
sc.pl.umap(
    mono_adata, 
    color=['file.batchID',  "subject.subjectGuid", 'leiden_0_5', 
           'subject.biologicalSex', 'AIFI_L2', 'AIFI_L3_new'],
    ncols=3, legend_loc = 'on data',
    wspace=0.4,
    save=  proj_name+'_rna_hamony_umap.png'
)

In [ ]:
# sc.pl.embedding(
#     mono_adata, basis='X_orignial_umap',
#     color=['file.batchID',  "subject.subjectGuid", 'leiden_0_5', 
#            'subject.biologicalSex', 'AIFI_L2', 'AIFI_L3_new'],
#     ncols=3, legend_loc = 'on data',
#     wspace=0.4,
#     save=  proj_name+'_rna_umap.png'
# )

In [ ]:
# save data
mono_adata.write_h5ad(data_path + 'ALTRA_scRNA_all_monocytes_certPro.h5ad')

### upload data to HISE

In [1]:
import hisepy as hp

In [2]:
# upload file to hise
hp.upload.upload_files(
    files=[
        "/home/workspace/data/ALTRA_manusript/ALTRA_scRNA_all_monocytes_certPro.h5ad"
    ],
    study_space_id="223de760-9624-45bd-aefe-ca24c75b1800",
    title="AlTRA scRNA monocyte h5ad object",
    input_file_ids=["609f7543-d4d5-41e9-a3d2-8e50c3e7c61d"], 
    destination='certpro'
)

{'Message': 'General Okay-ness',
 'VisualizationId': '00000000-0000-0000-0000-000000000000',
 'AbstractionId': '00000000-0000-0000-0000-000000000000',
 'TraceId': 'be253d7c-352d-4577-8d51-73d0a83fdd07',
 'ProcessId': '465d14cb-7665-4d6b-b10f-396c9b8503e5',
 'WorkflowId': '6082db0b-bcd2-4e96-a84a-d2f888081c89',
 'FileIds': ['f85270d8-5d91-4aa6-b8eb-887bd64e1f9e']}

## check expression

In [ ]:
# load the dataset
mono_adata = sc.read_h5ad(data_path + 'ALTRA_scRNA_all_monocytes_certPro.h5ad')

In [ ]:
mono_adata

In [ ]:
# sc.pl.umap(
#     mono_adata,
#     color=['file.batchID',  "subject.subjectGuid", 'leiden_0_5', 
#            'subject.biologicalSex', 'AIFI_L2', 'AIFI_L3'],
#     ncols=3,
#     wspace=0.4,
#     save=  proj_name+'_rna_hamony_umap.png'
# )

In [ ]:
with plt.rc_context({"figure.figsize": (8, 8), "figure.dpi": (100)}):
    sc.pl.umap(
    mono_adata,
    color=['leiden_1','AIFI_L3_new'], legend_loc='on data',
    ncols=2,
    save=  proj_name+'_rna_hamony_umap_leiden_1.png'
    )

In [ ]:
sc.pl.umap(
    mono_adata,
    color=['TNF', 'IL1B'],  vmin='p1',
    vmax='p99',
    frameon=False, 
    save=  proj_name+'_rna_hamony_umap_TNF.png'
)

In [ ]:
sc.tl.embedding_density(mono_adata, basis='umap', groupby='status')

In [ ]:
sc.pl.embedding_density(mono_adata, basis='umap', key='umap_density_status')

In [ ]:
mono_adata

In [ ]:
sc.pl.embedding(
    mono_adata,
    color=['AIFI_L3_new', 'leiden_1'], legend_loc='on data',
    basis='X_orignial_umap',
    frameon=False, ncols=2,
    save=  proj_name+'_celltype_leiden_1.pdf'
)

In [ ]:
sc.pl.embedding(
    mono_adata,
    color=['AIFI_L3_new', 'leiden_1', 'leiden_1_2', 'leiden_0_5'], legend_loc='on data',
    basis='X_tsne',
    frameon=False, ncols=2,
    save=  proj_name+'_celltype_leiden_1.pdf'
)

In [ ]:
sc.pl.embedding(
    mono_adata,
    color=['AIFI_L3_new', 'leiden_1'], legend_loc='on data',
    basis='X_tsne',
    frameon=False, ncols=2,
    save=  proj_name+'_celltype_leiden_1.pdf'
)

In [ ]:
sc.tl.embedding_density(mono_adata, basis='tsne', groupby='status')

In [ ]:
sc.pl.embedding_density(mono_adata, basis='tsne', key='tsne_density_status')

In [ ]:
mono_adata.obs['AIFI_L3_new'].unique()

In [ ]:
sc.pl.dotplot(mono_adata, ['TNF', 'IL1B', 'IL6', 'CCR2', 
                           'CD14','FCGR3A', 'HLA-DRA'], "pred_manual",
              save=  proj_name+'_rna_TNF_dotpolt.png',dendrogram=True)


In [ ]:
sc.pl.dotplot(mono_adata, ['TNF', 'IL1B', "IL1RN", 'CCR2', 
                           'CD14','FCGR3A'], "pred_manual", standard_scale='var',
              save=  proj_name+'_rna_TNF_dotpolt.png',dendrogram=True)


In [ ]:
mono_adata.obs.columns

# Analyze expression in paired comparison

In [ ]:
# load pair metadata 
pair_meta = pd.read_csv('/home/jupyter/data/ra_longitudinal/output_results/' + 'AIM3_paired_pre_post_conversion_samples.csv')
pair_meta.head()

In [ ]:
# subset the expression in paired comparison
pair_mono_adata = mono_adata[mono_adata.obs['sample.sampleKitGuid'].isin(pair_meta['sample.sampleKitGuid'])].copy()
pair_mono_adata

In [ ]:
pair_mono_adata.obs[['n_umis', 'total_counts']]

In [ ]:
# add status
pair_mono_adata.obs.drop(columns='status', inplace=True)
pair_mono_adata.obs = pair_mono_adata.obs.merge(pair_meta[['sample.sampleKitGuid','status']], how='left', on='sample.sampleKitGuid')
pair_mono_adata

In [ ]:
# load data
pair_mono_adata = sc.read_h5ad(data_path + 'ALTRA_scRNA_monocytes_paired_certPro.h5ad')

In [ ]:
# pair_meta[['sample.sampleKitGuid','status']]

In [ ]:
pair_mono_adata.obs['AIFI_L3_new'].value_counts()

In [ ]:
pair_mono_adata.obs['celltype_status'] = pair_mono_adata.obs['AIFI_L3_new'].astype('str') + pair_mono_adata.obs['status'].astype('str')

In [ ]:
sc.pl.dotplot(pair_mono_adata, ['TNF', 'IL1B', "IL1RN", 'CCR2', 
                           'CD14','FCGR3A'], "celltype_status", standard_scale='var',
              save=  proj_name+'_paired_samples_rna_TNF_dotpolt.png',dendrogram=True)


In [ ]:
pair_mono_adata

In [ ]:
sc.pl.dotplot(pair_mono_adata, ['TNF', 'IL1B', "IL1RN", 'CCR2', 
                           'CD14','FCGR3A'], "celltype_status", standard_scale='var',
              save=  proj_name+'_paired_samples_rna_TNF_dotpolt_status.png',dendrogram=True)

In [ ]:
# plot the markers gene list from the paper
gene_list = ['NFKBIA', 'TNF', 'IL1B', "CCL3", 'CCL4', 'ICAM1', 'FOLR2', 
            'CD14', 'FCGR3A', 'KLF6', 'NR4A1', 'DUSP1', 'ATF3']

sc.pl.dotplot(pair_mono_adata, gene_list, "AIFI_L3",  standard_scale='var', swap_axes=True,
              save=  proj_name+'_AIFI_L3_target_genes.pdf')


In [ ]:
with plt.rc_context({"figure.figsize": (8, 4)}):
    sc.pl.violin(pair_mono_adata, ["TNF"], groupby="AIFI_L3", 
                 save=  proj_name+'_rna_TNF_violin.png',
                 inner="box", rotation=90)

In [ ]:
# setting highly variable as highly deviant to use scanpy 'use_highly_variable' argument in sc.pp.pca
sc.pp.pca(pair_mono_adata, svd_solver="arpack", use_highly_variable=True)
# plot the principle component variance explained
sc.pl.pca_variance_ratio(pair_mono_adata, log=True)

In [ ]:
pair_mono_adata

In [ ]:
sc.pl.embedding(
    pair_mono_adata,
    color=['AIFI_L3_new', 'leiden_1'], legend_loc='on data',
    basis='X_tsne',
    frameon=False, ncols=2,
    save=  proj_name+'_paired_leiden_1.pdf'
)

In [ ]:
# set order for status column
pair_mono_adata.obs['status'] = pair_mono_adata.obs['status'].astype('str')
pair_mono_adata.obs.loc[pair_mono_adata.obs['status']=='pre_conv', 'status'] = 'pre-disease'
pair_mono_adata.obs['status'] = pair_mono_adata.obs['status'].astype('category').cat.reorder_categories(
    ['pre-disease', 'conversion'], ordered=True)

In [ ]:
sc.tl.embedding_density(pair_mono_adata, basis='tsne', groupby='status')

In [ ]:
sc.set_figure_params(scanpy=True, fontsize=14) 
sc.pl.embedding_density(pair_mono_adata, basis='tsne',
                        key='tsne_density_status', 
                        ncols=2, 
                        color_map= 'magma',
                       save=  proj_name+'_paired_tsne_density_status.pdf')

In [ ]:
sc.pl.embedding(
    pair_mono_adata,
    color=['status', 'leiden_1', 'AIFI_L3_new'],#legend_loc='on data',
    basis='X_tsne',
    save=  proj_name+'_paired_status.png'
)

In [ ]:
with plt.rc_context({"figure.figsize": (4, 4), "figure.dpi": (400)}):
        ax = sc.pl.embedding(
        pair_mono_adata,
        color=['AIFI_L3_new'],  #legend_loc='on data', 
        legend_fontsize='small',
        frameon=False, basis='X_tsne',
        save = proj_name+'_paired_AIFI_L3_new.pdf'
    )


In [ ]:
sc.pl.embedding(
    pair_mono_adata,
    color=['leiden_1'], legend_loc='on data',
    frameon=False, basis='X_tsne',
    save = proj_name+'_paired_leiden_1.pdf'
)

In [ ]:
sc.pl.embedding(
    pair_mono_adata,
    color=['TNF', 'IL1B', 'CCL3', 'NFKBIA'],#legend_loc='on data',
    basis='X_tsne',
        vmin='p1',
    vmax='p99',
    frameon=False, ncols=2,
    save=  proj_name+'_marker_genes.png'
)

In [ ]:
sc.pl.violin(pair_mono_adata, ['TNF'], groupby='AIFI_L3', rotation=90, save=proj_name+'TNF_violinplot.png')

In [ ]:
pair_mono_adata

In [ ]:
# save data
pair_mono_adata.write_h5ad(data_path + 'ALTRA_scRNA_monocytes_paired_certPro.h5ad')

### analyze the clusters

In [ ]:
# save data
pair_mono_adata = sc.read_h5ad(data_path + 'ALTRA_scRNA_monocytes_paired_certPro.h5ad')

### plot Fig 2I

In [ ]:
gene_list = ['IL1B', 'TNF', "CCL3", 'CCL4','CXCL3', 'CXCL8','CXCL10', 'ICAM1', 'FOLR2' ,'NFKBIA', 'NLRP3',
            'CD14', 'FCGR3A', 'HLA-DRA', 'CCR2', 'KLF6', 'NR4A1', 'DUSP1', 'ATF3']
cm = 1/2.54  # centimeters in inches

dp=sc.pl.dotplot(pair_mono_adata, gene_list, "AIFI_L3", standard_scale='var', figsize=[10.73/1.2, 3.21/1.2],
               #  return_fig=True,
              save = proj_name+'_paired_samples_AIFI_L3.pdf',  
              dendrogram=False)

In [ ]:
# Create the dotplot and return the figure
dotplot = sc.pl.dotplot(pair_mono_adata, gene_list, groupby="AIFI_L3", standard_scale='var', 
                        dendrogram=False, return_fig=True)
# Extract the figure and axes from the DotPlot object
axes = dotplot.get_axes()

In [ ]:
gene_list = ['NFKBIA', 'TNF', 'IL1B', "CCL3", 'CCL4', 'ICAM1', 'FOLR2', 
            'CD14', 'FCGR3A', 'KLF6', 'NR4A1', 'DUSP1', 'ATF3']
IL1B, TNF, CCL3, CCL4, CXCL3, CXCL8
sc.pl.dotplot(pair_mono_adata, gene_list, "leiden_1", standard_scale='var',
              save = proj_name+'_paired_samples_leiden_1_dotpolt.png', 
              dendrogram=True)


## compare IL1b monocyte with the core CD14 monocyte

In [ ]:
pair_mono_adata.obs['AIFI_L3'].unique()


In [ ]:
# test for top genes that seperate the cells
cluster_name = 'AIFI_L3'
sc.tl.rank_genes_groups(pair_mono_adata, groupby=cluster_name, 
                        groups= ['IL1B+ CD14 monocyte'], reference='Core CD14 monocyte',
                        method='wilcoxon', key_added='wilcoxon_il1b_vs_core_cd14')

In [ ]:
il1bvscore_cd14_degs = sc.get.rank_genes_groups_df(pair_mono_adata, key= 'wilcoxon_il1b_vs_core_cd14',
                                              pval_cutoff=0.05,  group=None)
il1bvscore_cd14_degs = il1bvscore_cd14_degs.reindex(il1bvscore_cd14_degs['pvals_adj'].sort_values(ascending=True).index)
il1bvscore_cd14_degs.to_csv(output_path + proj_name + cluster_name+'_il1bvscore_cd14_wilcoxon_sig_degs.csv')

In [ ]:
cluster_name = 'AIFI_L3'
il1bvscore_cd14_degs= pd.read_csv(output_path + proj_name + cluster_name+'_il1bvscore_cd14_wilcoxon_sig_degs.csv')

In [ ]:
il1bvscore_cd14_degs = il1bvscore_cd14_degs.set_index('names')
il1bvscore_cd14_degs

In [ ]:
plot_volcano_df(
    il1bvscore_cd14_degs,
    x='logfoldchanges',
    y='pvals_adj',
    #genes=['IL1B', 'CCL3', 'CCL4', 'BTG2', 'EGR1', 'NFKB1A', 'TNF'], 
    lFCs_thr = 0.1, 
    sign_thr = 0.01,
    figsize=(5, 5), dpi=500,
    save= fig_path + proj_name+ cluster_name+ '_il1bvscore_cd14_degs_top40_valcano.pdf'
    )


In [ ]:
# test for top genes that seperate the cells
cluster_name = 'leiden_1'
sc.tl.rank_genes_groups(pair_mono_adata, groupby=cluster_name,
                        method='wilcoxon', key_added='leiden_1_wilcoxon')

In [ ]:
sc.pl.rank_genes_groups(pair_mono_adata, n_genes=25, sharey=False, key="leiden_1_wilcoxon")

In [ ]:
sc.pl.rank_genes_groups_dotplot(
    pair_mono_adata, groupby=cluster_name, standard_scale="var", 
    n_genes=10, key=cluster_name +"_wilcoxon",
    save= '_'+ proj_name+ cluster_name + '_wilcoxon_top_genes_dotplot_scale.png'
)

In [ ]:
# output the deg list 
leiden_deg = sc.get.rank_genes_groups_df(pair_mono_adata, key= cluster_name + '_wilcoxon',
                                              pval_cutoff=None,  group=None).rename(
    {'group':cluster_name},  axis='columns')
leiden_deg['direction'] = np.where(leiden_deg['logfoldchanges']>0, 'up', 'down')
leiden_deg = leiden_deg.reindex(leiden_deg['scores'].abs().sort_values(ascending=False).index)
leiden_deg.to_csv(output_path + proj_name + cluster_name+'_wilcoxon_degs.csv')

### compare the il1b monocytes with the tissue macrophage genes

In [ ]:
# load the degs table
macrophage_degs = pd.read_excel('/home/jupyter/data/ra_longitudinal/reference_data/Alivernini_et_al_tissue_Macrophage' + 
                             '/41591_2020_939_MOESM3_ESM_cluster_degs.xlsx')
# take cluster 8 
cluster_degs_c8 = macrophage_degs.loc[macrophage_degs['cluster'] == 8]
cluster_degs_c8

In [ ]:
cluster_degs_c8['average log foldchange'].hist()

In [ ]:
# cluster_degs_c8.loc[cluster_degs_c8['average log foldchange']>0]

In [ ]:
c8_degs = cluster_degs_c8.loc[cluster_degs_c8['average log foldchange']>0, 'gene symbol']
len(c8_degs)

In [ ]:
c8_degs

In [ ]:
# create a C8 scores
sc.tl.score_genes(pair_mono_adata, c8_degs, score_name='c8_score')

In [ ]:
pair_mono_adata.obs[['c8_score', 'AIFI_L3_new', 'sample.sampleKitGuid','subject.subjectGuid', 'subject.biologicalSex', 
                              'status']].to_csv(output_path + 'ALTRA_mono_C8_scores.csv')

In [ ]:
output_path + 'ALTRA_mono_C8_scores.csv'

In [ ]:
sns.set_style("ticks")
with plt.rc_context({"figure.figsize": (4, 4)}):
    sc.pl.violin(
        pair_mono_adata,
        ["c8_score"],
        groupby="AIFI_L3_new",
        rotation=90,
        stripplot=False,  # remove the internal dots
        inner="box",  # adds a boxplot inside violins
        save = proj_name + 'C8_AIFI_L3_score.pdf'
    )
#sc.pl.violin(pair_mono_adata, 'c8_score', stripplot=False, groupby='AIFI_L3_new')

In [ ]:
# from scipy.stats import f_oneway
# c8_scores = pair_mono_adata.obs[['c8_score', 'AIFI_L3_new', 'sample.sampleKitGuid']]
# celltypes = c8_scores['AIFI_L3_new'].unique().tolist()
# celltypes

In [ ]:
# from scipy.stats import f_oneway
# c8_scores = pair_mono_adata.obs[['c8_score', 'AIFI_L3_new']]
# celltypes = c8_scores['AIFI_L3_new'].unique()
# # Perform ANOVA
# f_statistic, p_value = f_oneway(c8_scores[c8_scores['AIFI_L3_new'] == celltypes[0]]['c8_score'], 
#                                 c8_scores[c8_scores['AIFI_L3_new'] == celltypes[1]]['c8_score'],
#                                 c8_scores[c8_scores['AIFI_L3_new'] == celltypes[2]]['c8_score'],
#                                 c8_scores[c8_scores['AIFI_L3_new'] == celltypes[3]]['c8_score'],
#                                 c8_scores[c8_scores['AIFI_L3_new'] == celltypes[4]]['c8_score'],
#                                 c8_scores[c8_scores['AIFI_L3_new'] == celltypes[5]]['c8_score'],
#                                 c8_scores[c8_scores['AIFI_L3_new'] == celltypes[6]]['c8_score'])

# print("F-statistic:", f_statistic)
# print("p-value:", p_value)

In [ ]:
# from statsmodels.stats.multicomp import pairwise_tukeyhsd

# # Perform Tukey's HSD test
# tukey = pairwise_tukeyhsd(endog=c8_scores['c8_score'], groups=c8_scores['AIFI_L3_new'], alpha=0.05)
# print(tukey)

In [ ]:
pair_mono_sub = pair_mono_adata[:, pair_mono_adata.var.index.isin(macrophage_degs['gene symbol'])].copy()
pair_mono_sub

In [ ]:
# setting highly variable as highly deviant to use scanpy 'use_highly_variable' argument in sc.pp.pca
sc.pp.pca(pair_mono_sub, svd_solver="arpack")
# plot the principle component variance explained
sc.pl.pca_variance_ratio(pair_mono_sub, log=True)

In [ ]:
pair_mono_sub

In [ ]:
sc.pp.neighbors(pair_mono_sub, n_neighbors=10, n_pcs=5)
sc.tl.umap(pair_mono_sub)

In [ ]:
# sc.pp.neighbors(mono_adata, n_neighbors=10, n_pcs=30, use_rep='X_pca_harmony')
sc.tl.tsne(pair_mono_sub)

In [ ]:
sc.pl.embedding(
    pair_mono_sub, palette='tab10',
    color=['AIFI_L3_new'], legend_loc='on data', size=6,
    basis='X_tsne', 
    frameon=False, 
    save=proj_name+'_AIFI_L3_new.pdf'
)

In [ ]:
sc.pl.umap(
    pair_mono_sub,
    color=['AIFI_L3'], size=6,
    save=  proj_name+'_rna_umap.png'
)

In [ ]:
# take marker genes for c8
c8_genes = cluster_degs_c8['gene symbol'][0:10][cluster_degs_c8['gene symbol'][0:10].isin(pair_mono_adata.var.index)]
c8_genes

In [ ]:
gene_list = ['CD14', 'FCGR3A', 'HLA-DRA', 'CCR2', 
             'IL1B', 'TNF', "CCL3", 'CCL4','CXCL3', 'CXCL8', 'ICAM1', 'NFKBIA', 'NLRP3',
             'KLF6', 'NR4A1', 'DUSP1']
cm = 1/2.54  # centimeters in inches

dp=sc.pl.dotplot(pair_mono_adata, gene_list, "AIFI_L3", standard_scale='var', 
                 figsize=[10.73/1.2, 3.21/1.2],
               #  return_fig=True,
              save = proj_name+'_paired_samples_AIFI_L3.pdf',  
              dendrogram=False)

In [ ]:
gene_list = pd.Series(['CD14', 'FCGR3A', 'HLA-DRA', 'CCR2', 
             'IL1B', 'TNF', "CCL3", 'CCL4','CXCL3', 'CXCL8', 'CXCL10', 'ICAM1', 'NFKBIA', 'NLRP3',
             'KLF6', 'NR4A1', 'DUSP1'])
gene_list[~gene_list.isin(c8_degs)]

In [ ]:
# Create the dotplot and return the figure
sc.pl.dotplot(pair_mono_adata, c8_genes,
              groupby="AIFI_L3", standard_scale='var')

## extract frequency by L3

In [ ]:
# extract the frequency out of the monocytes
cluster_name = 'AIFI_L3_new'
cell_counts = pair_mono_adata.obs.groupby(['sample.sampleKitGuid',cluster_name]).size().reset_index(name='counts').rename(
    {cluster_name:'cell_type'}, axis=1)
meta_keep = pair_mono_adata.obs.loc[:, 
                            ['sample.sampleKitGuid','subject.subjectGuid', 'subject.biologicalSex', 
                              'status', 'file.batchID']].drop_duplicates()
total_mono_counts = pair_mono_adata.obs.groupby(['sample.sampleKitGuid']).size().reset_index(name='total_mono_counts')
mono_freq = cell_counts.merge(meta_keep,  how='left',on=['sample.sampleKitGuid']).merge(total_mono_counts, 
                                            how='left',on=['sample.sampleKitGuid'])
mono_freq['mono_frequency'] = mono_freq['counts']/mono_freq['total_mono_counts']
mono_freq

In [ ]:
mono_freq.to_csv(output_path + proj_name + 'paired_mono_'+cluster_name+'_frequency.csv')

In [ ]:
output_path + proj_name + 'paired_mono_' + cluster_name+'_frequency.csv'

## subset the cd16 monocytes

In [ ]:
mono_adata.obs['pred_manual'].unique()

In [ ]:
cd16_mono = mono_adata[
    (mono_adata.obs['pred_manual'].str.contains('CD16|Int|IL1B'))].copy()
cd16_mono

In [ ]:
# save the dataset
cd16_mono.write_h5ad(data_path+'ALTRA_scRNA_CD16_monocytes.h5ad')

# Session Info

In [ ]:
import sinfo
sinfo.sinfo(write_req_file = False)